# Ijeoma AI — Train and EvaluateCompanion to **Lab 8.9**, wired to the Ijeoma dataset and the grounded taskdesign: every prompt carries its passage, so the model reads rather thanrecalls.Three things this notebook does that the lab template does not:1. **Scores the untuned model first.** Same prompts, same test set, same   metrics. Without that number you cannot say fine-tuning helped.2. **Scores per task.** `classify`, `answer`, `summarize` and `refuse` fail in   different ways; one averaged number hides all of it.3. **Measures the invention rate.** Prompts about places that are not in the   corpus, where the only correct answer is to decline.---### Run the cells in order, top to bottomTwo rules, both of which cost a run when broken:1. **Cell 1 must run before anything imports Keras.** The backend is fixed at   import time. If you have already imported Keras, go to   **Runtime → Restart session** and start again from cell 1.2. **Runtime → Change runtime type → T4 GPU** before you start.`SMOKE_TEST = True` is the default: 200 examples, 1 epoch, end to end inroughly fifteen minutes. Get a clean pass, then set it to `False`.

## 1. Setup — run this first

In [1]:
# Keras reads KERAS_BACKEND when it is first imported, so this must happen# before any import of keras, keras_hub or keras_nlp. If Keras is already in# sys.modules the backend is locked in and only a restart clears it.import sysif "keras" in sys.modules:    raise RuntimeError(        "Keras is already imported, so the backend is locked. "        "Runtime -> Restart session, then run this cell first."    )import osos.environ["KERAS_BACKEND"] = "jax"os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"print("backend requested: jax")

In [2]:
!pip install -q -U keras-hub keras

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.33.0 which is incompatible.


In [3]:
import jsonimport randomimport reimport timefrom collections import Counterimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport kerasimport keras_hub as khprint("keras      ", keras.__version__)print("keras_hub  ", kh.__version__)print("backend    ", keras.backend.backend())if keras.backend.backend() != "jax":    print("\nWARNING: backend is not jax. Something imported keras before "          "cell 1 ran. Runtime -> Restart session and start again from the top.")!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU - Runtime > Change runtime type > T4"

SyntaxError: invalid syntax (2336129259.py, line 1)

### CredentialsGemma weights live behind a licence gate. You need **one** of these working;the loader tries them in order and tells you which one succeeded.**Kaggle (the route the course uses)**1. Accept the licence on the [Gemma 3 Keras model   page](https://www.kaggle.com/models/keras/gemma3/keras/) — *Request   Access*, read, accept. Nothing below works until this is done, and a   missing acceptance looks exactly like a wrong key: HTTP 403.2. Kaggle → profile icon → **Settings** → **API** → **Create New Token**,   which downloads `kaggle.json` containing a username and a key.3. In Colab, 🔑 in the left sidebar → add `KAGGLE_USERNAME` and `KAGGLE_KEY`,   values pasted **without quotes**, **Notebook access ON** for both.**Hugging Face (the fallback)**1. Accept the licence on   [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it).2. Settings → Access Tokens → create a **read** token.3. Add it as a Colab secret named `HF_TOKEN`, notebook access ON.> **What went wrong last time.** The previous version had a> `WEIGHTS_SOURCE = "hf"` switch that loaded an HF token but left the preset> string as `gemma3_instruct_1b` — and a bare preset name always resolves to> Kaggle. So the request went to Kaggle with no credentials and returned 403.> Switching source now switches the actual URI, and the loader falls through> the alternatives rather than failing on the first one.

In [ ]:
from google.colab import userdata, drivedef get_secret(name):    try:        v = userdata.get(name)        return v.strip() if v else None    except Exception:        return NoneHAVE_KAGGLE = FalseKAGGLE_USERNAME = get_secret("KAGGLE_USERNAME")KAGGLE_KEY = get_secret("KAGGLE_KEY")if KAGGLE_USERNAME and KAGGLE_KEY:    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME    os.environ["KAGGLE_KEY"] = KAGGLE_KEY    HAVE_KAGGLE = True    print(f"Kaggle credentials loaded (user: {KAGGLE_USERNAME})")else:    print("Kaggle credentials NOT found")HAVE_HF = FalseHF_TOKEN = get_secret("HF_TOKEN")if HF_TOKEN:    os.environ["HF_TOKEN"] = HF_TOKEN    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN    HAVE_HF = True    print("Hugging Face token loaded")else:    print("Hugging Face token NOT found")if not (HAVE_KAGGLE or HAVE_HF):    print("\nNeither credential is set. Add one in the Secrets panel (key icon, "          "left sidebar), switch Notebook access on, and re-run this cell.")drive.mount("/content/drive")

## 2. ConfigEvery knob in one place, so the experiment log at the bottom records what youactually ran.**On the preset.** The lab template loads `gemma3_1b`, the pretrained basemodel, and argues it is a more neutral starting point for LoRA. Fair, but itmakes the baseline comparison empty: a base model has never been trained tofollow an instruction, so of course it rambles, and "fine-tuning beat it"tells you nothing. Ijeoma's tasks are reading tasks and instruction-followingis most of the job, so `gemma3_instruct_1b` is the default.Note there is no `_en` suffix on Gemma 3 presets — `gemma3_instruct_1b_en`does not exist, and the error it produces ("no config.json") looks like apermissions problem rather than a typo.

In [ ]:
# ---- data -----------------------------------------------------------------DRIVE = "/content/drive/MyDrive/Google Deep mind/Google Deepmind Tour Guide IJEOMA"TRAIN_PATH = f"{DRIVE}/conversational_training_data.jsonl"TEST_PATH = f"{DRIVE}/conversational_test_data.jsonl"# The _full files carry task/subtype/country/passage_id. Without them the# notebook falls back to reading the task from each prompt's tag, which works# but gives up the per-country breakdown and the leakage check.TRAIN_FULL_PATH = f"{DRIVE}/pairs_train_full.jsonl"TEST_FULL_PATH = f"{DRIVE}/pairs_test_full.jsonl"# "instruction" = classify/answer/refuse, "summarization" = the extractive# summarization set, "both" = concatenated. The two sets share one# passage-level split, so combining them does not leak.DATASET = "both"SUM_TRAIN_PATH = f"{DRIVE}/summarization_train_full.jsonl"SUM_TEST_PATH = f"{DRIVE}/summarization_test_full.jsonl"# ---- model ----------------------------------------------------------------# Tried in order; the first that loads wins. A bare name resolves to Kaggle.PRESET_CANDIDATES = [    "gemma3_instruct_1b",              # Kaggle (needs KAGGLE_* + licence)    "hf://google/gemma-3-1b-it",       # Hugging Face transformers checkpoint    "hf://keras/gemma3_instruct_1b",   # Keras mirror on Hugging Face]# ---- run size -------------------------------------------------------------SMOKE_TEST = TrueTRAIN_SUBSET = 200EVAL_N = 20ADVERSARIAL_N = 20# ---- LoRA -----------------------------------------------------------------LORA_RANK = 8LEARNING_RATE = 2e-4NUM_EPOCHS = 1 if SMOKE_TEST else 2BATCH_SIZE = 1SEQUENCE_LENGTH = None      # measured from the data in section 5GEN_MAX_LENGTH = None       # derived from SEQUENCE_LENGTH# ---- misc -----------------------------------------------------------------SEED = 812MIXED_PRECISION = FalsePRESET = None               # set by the loaderkeras.utils.set_random_seed(SEED)random.seed(SEED)print("smoke test:", SMOKE_TEST, "| dataset:", DATASET)

**On mixed precision.** It roughly halves memory, and the T4 is a Turing cardwith no native bfloat16 — so if you enable it, it must be `mixed_float16`,never `mixed_bfloat16`. Off by default because float16 can produce NaN losseswith LoRA on JAX, and a silent NaN is a bad way to spend an evening.

In [ ]:
if MIXED_PRECISION:    keras.mixed_precision.set_global_policy("mixed_float16")    print("mixed precision ON - watch for NaN loss")else:    print("mixed precision off (float32)")

## 3. Load the data

In [ ]:
def load_jsonl(path):    with open(path, encoding="utf-8") as fh:        return pd.DataFrame([json.loads(l) for l in fh if l.strip()])def infer_task(prompt):    '''Read the task off the prompt tag.    Lets per-task scoring work even when the _full metadata files are not on    Drive - the tag is written into every input by the data builder.    '''    head = str(prompt)[:40].upper()    if head.startswith("[CLASSIFY]"):        return "classify"    if head.startswith("[SUMMARIZE]"):        return "summarize"    if head.startswith("[DIGEST]"):        return "digest"    if head.startswith("[ANSWER]"):        # Refusals arrive as ordinary questions; the gold output is what        # distinguishes them, and only the _full files know for sure.        return "answer"    return "unknown"for label, path in (("train", TRAIN_PATH), ("test", TEST_PATH),                    ("train_full", TRAIN_FULL_PATH), ("test_full", TEST_FULL_PATH),                    ("sum_train", SUM_TRAIN_PATH), ("sum_test", SUM_TEST_PATH)):    print(f"{'found  ' if os.path.exists(path) else 'MISSING'}  {label:11s} {os.path.basename(path)}")

In [ ]:
train_df = load_jsonl(TRAIN_PATH)test_df = load_jsonl(TEST_PATH)HAVE_META = os.path.exists(TRAIN_FULL_PATH) and os.path.exists(TEST_FULL_PATH)if HAVE_META:    train_full, test_full = load_jsonl(TRAIN_FULL_PATH), load_jsonl(TEST_FULL_PATH)else:    print("_full files not on Drive - falling back to the prompt tag for task "          "labels. Per-task scoring still works; the per-country breakdown and "          "the leakage assertion do not. Upload pairs_train_full.jsonl and "          "pairs_test_full.jsonl from the dataset bundle to get them back.\n")    train_full, test_full = train_df.copy(), test_df.copy()    for d in (train_full, test_full):        d["task"] = d["input"].map(infer_task)        d["subtype"] = None        d["country"] = None        d["passage_id"] = None# Merge the summarization set when asked. Both were split on the same passage# ids, so concatenating cannot put one passage on both sides.if DATASET in ("summarization", "both"):    if os.path.exists(SUM_TRAIN_PATH) and os.path.exists(SUM_TEST_PATH):        s_tr, s_te = load_jsonl(SUM_TRAIN_PATH), load_jsonl(SUM_TEST_PATH)        if DATASET == "summarization":            train_full, test_full = s_tr, s_te        else:            train_full = pd.concat([train_full, s_tr], ignore_index=True)            test_full = pd.concat([test_full, s_te], ignore_index=True)        print(f"summarization merged: +{len(s_tr)} train, +{len(s_te)} test")    else:        print("summarization _full files not found - continuing without them")train_data = train_full[["input", "output"]].to_dict("records")test_data = test_full[["input", "output"]].to_dict("records")print(f"\ntrain {len(train_data)}   test {len(test_data)}   metadata {HAVE_META}")print("\ntest set by task:")print(test_full["task"].value_counts().to_string())

In [ ]:
# Leakage check. Only meaningful with the _full files, which carry passage_id.if HAVE_META and test_full["passage_id"].notna().any():    tr = set(train_full["passage_id"].dropna())    te = set(test_full["passage_id"].dropna())    overlap = tr & te    print(f"train passages {len(tr)} | test passages {len(te)} | overlap {len(overlap)}")    assert not overlap, f"LEAKAGE: {len(overlap)} passages in both splits"    print("no leakage")else:    print("skipped - needs the _full files")

## 4. Load GemmaLoaded before the sequence-length measurement so the tokenizer comes from themodel itself — one download, one failure point, and no chance of thetokenizer and the model disagreeing.

In [ ]:
def load_gemma(candidates):    '''Try each preset in turn; return the first that loads.'''    errors = []    for preset in candidates:        if preset.startswith("hf://") and not HAVE_HF:            errors.append((preset, "skipped - no HF_TOKEN"))            continue        if not preset.startswith("hf://") and not HAVE_KAGGLE:            errors.append((preset, "skipped - no Kaggle credentials"))            continue        try:            print(f"trying {preset} ...")            t0 = time.time()            m = kh.models.Gemma3CausalLM.from_preset(preset)            print(f"  loaded in {time.time() - t0:.0f}s")            return m, preset        except Exception as exc:            msg = str(exc).split("\n")[0][:160]            errors.append((preset, msg))            print(f"  failed: {msg}")    print("\nNone of the presets loaded:\n")    for p, e in errors:        print(f"  {p}\n     {e}")    print("\n403 from Kaggle almost always means the Gemma licence has not "          "been accepted on your account, not that the key is wrong.")    raise RuntimeError("could not load any Gemma preset - see the list above")model, PRESET = load_gemma(PRESET_CANDIDATES)RUN_NAME = f"{PRESET.split('/')[-1]}-r{LORA_RANK}-lr{LEARNING_RATE}-e{NUM_EPOCHS}"print("\npreset:", PRESET)print("run   :", RUN_NAME)

## 5. Measure the sequence lengthThe lab template sets `sequence_length = 450`. Ijeoma's prompts carry a wholepassage, so 450 would cut most of them in half — and a truncated passage isworse than no passage: the model is trained to answer a question whoseevidence has been deleted, which is a direct lesson in making things up.

In [ ]:
tokenizer = model.preprocessor.tokenizerSOT, EOT = "<start_of_turn>", "<end_of_turn>"def full_text(rec):    return f"{SOT}user\n{rec['input']}{EOT}{SOT}model\n{rec['output']}{EOT}"sample = random.sample(train_data, min(400, len(train_data)))lengths = np.array([len(tokenizer.tokenize(full_text(r))) for r in sample])p50, p95, p99, mx = (int(np.percentile(lengths, q)) for q in (50, 95, 99, 100))print(f"tokens  median {p50}  p95 {p95}  p99 {p99}  max {mx}")if SEQUENCE_LENGTH is None:    SEQUENCE_LENGTH = int(min(1024, 64 * np.ceil((mx + 16) / 64)))if GEN_MAX_LENGTH is None:    # Room for the prompt plus an answer. Too small and long prompts leave no    # tokens to answer in, which looks like a broken model.    GEN_MAX_LENGTH = int(min(1536, SEQUENCE_LENGTH + 256))model.preprocessor.sequence_length = SEQUENCE_LENGTHtruncated = float((lengths > SEQUENCE_LENGTH).mean())print(f"\nSEQUENCE_LENGTH = {SEQUENCE_LENGTH}   GEN_MAX_LENGTH = {GEN_MAX_LENGTH}")print(f"{truncated:.1%} of examples truncated")if truncated > 0.01:    print("WARNING: raise SEQUENCE_LENGTH, or shorten passages when building "          "the data. If the T4 runs out of memory at this length, shorten the "          "passages - do not silently truncate here.")

In [ ]:
plt.figure(figsize=(7, 3))plt.hist(lengths, bins=40, color="#6b7280")plt.axvline(SEQUENCE_LENGTH, color="#2563eb", linestyle="--",            label=f"sequence_length = {SEQUENCE_LENGTH}")plt.xlabel("tokens per training example")plt.ylabel("examples")plt.title("Nothing should sit to the right of the line")plt.legend()plt.tight_layout()plt.show()

## 6. Format as turnsWorth knowing what the dict does: the Gemma preprocessor takes`{"prompts": ..., "responses": ...}` and returns `(x, y, sample_weight)` with`sample_weight` 0 on prompt tokens and 1 on response tokens — so **the loss isalready computed on the response only**. You do not need to mask it yourself,and you should not pass one concatenated string, which would train the modelto generate the passage back at you.

In [ ]:
def format_io_as_turns(input_text, output_text, sot=SOT, eot=EOT):    '''Wrap one pair as a user turn and a model turn.'''    return f"{sot}user\n{input_text}{eot}", f"{sot}model\n{output_text}{eot}"fit_records = train_data[:TRAIN_SUBSET] if SMOKE_TEST else train_dataprompts, responses = [], []for rec in fit_records:    q, a = format_io_as_turns(rec["input"], rec["output"])    prompts.append(q)    responses.append(a)data = {"prompts": prompts, "responses": responses}print(f"{len(prompts)} training examples prepared")print("\n--- prompt ---\n", prompts[0][:400])print("\n--- response ---\n", responses[0][:300])

In [ ]:
def generate_answer(question, max_length=None):    '''One stateless generation. Returns the model text only.'''    prompt = f"{SOT}user\n{question}{EOT}\n{SOT}model\n"    raw = model.generate(prompt, max_length=max_length or GEN_MAX_LENGTH)    out = raw[len(prompt):] if raw.startswith(prompt) else raw.replace(prompt, "")    out = out.strip()    if out.endswith(EOT):        out = out[: -len(EOT)].strip()    for tok in (SOT + "user", SOT + "model", "<eos>", "<pad>"):        idx = out.find(tok)        if idx > 0:            out = out[:idx].strip()    return outprint(generate_answer("[ANSWER] Answer using only the passage.\n\n"                      "QUESTION: What is this place?\n\nPASSAGE:\n"                      "Elmina Castle is a fortification on the coast of Ghana, "                      "built by the Portuguese in 1482.")[:300])

## 7. ScoringDefined once and run against both models, so the comparison is like for like.| Task | What is measured || --- | --- || `classify` | JSON parses, then exact match per field and macro-F1 on theme || `answer` / `digest` / `summarize` | Grounding — every number and proper noun present in the passage — plus ROUGE-L against the reference || `refuse` | Did it refuse when it should have, **and** did it refuse when it should not have |Grounding is an automatic proxy, not the 90% claim-support figure from theproposal — that needs humans reading claims against passages. It tracks thesame failure cheaply enough to run after every change.

In [ ]:
RE_NUM = re.compile(r"\d[\d,.]*")RE_PROPER = re.compile(r"\b[A-Z][a-zÀ-ɏ'’\-]{2,}\b")# Verified against the gold refusals in the dataset: recall 1.00, zero false# positives on gold answers. If you add refusal phrasings when you rebuild the# data, re-check this regex against them or the metric drifts silently.RE_REFUSAL = re.compile(    r"((?:does|do)(?:\s+not|n't)\s+"    r"(?:say|mention|cover|provide|describe|give|list|include|state|have)"    r"|i (?:cannot|can't|am not able|do not have|don't have)"    r"|cannot advise|can't advise|unable to|no information"    r"|not covered|not mentioned|not stated"    r"|source does not|passage does not)", re.I)STOPWORD_PROPER = {    "The", "This", "That", "It", "They", "There", "A", "An", "In", "On", "At",    "If", "When", "What", "Which", "Where", "How", "You", "Your", "I", "We",    "Check", "Passage", "Its", "His", "Her", "None", "True", "False",}GROUNDED_TASKS = ("answer", "digest", "summarize")def norm_numbers(text):    out = set()    for n in RE_NUM.findall(text):        n = n.rstrip(".,").replace(",", "")        if n:            out.add(n)    return outdef extract_passage(inp):    i = inp.find("PASSAGE:")    return inp[i + len("PASSAGE:"):].strip() if i >= 0 else ""def is_grounded(output, passage):    '''True when every number and proper noun in the output is in the passage.'''    if not passage or passage == "(none available)":        return None    if not norm_numbers(output) <= norm_numbers(passage):        return False    low = passage.lower()    unseen = [w for w in RE_PROPER.findall(output)              if w not in STOPWORD_PROPER and w.lower() not in low]    return len(unseen) <= 1def lcs_length(a, b):    prev = [0] * (len(b) + 1)    for x in a:        cur = [0]        for j, y in enumerate(b):            cur.append(prev[j] + 1 if x == y else max(cur[j], prev[j + 1]))        prev = cur    return prev[-1]def rouge_l(pred, ref):    p, r = pred.lower().split(), ref.lower().split()    if not p or not r:        return 0.0    l = lcs_length(p, r)    if l == 0:        return 0.0    prec, rec = l / len(p), l / len(r)    return 2 * prec * rec / (prec + rec)def parse_labels(text):    try:        return json.loads(re.sub(r"^```(?:json)?|```$", "", text.strip(),                                 flags=re.M).strip())    except json.JSONDecodeError:        m = re.search(r"\{.*\}", text, re.S)        if m:            try:                return json.loads(m.group(0))            except json.JSONDecodeError:                return None        return Noneprint("scoring helpers defined")

In [ ]:
def evaluate(records, label, n=None, verbose_examples=2):    '''Run the model over records and score per task. Returns (summary, rows).'''    items = records[:n] if n else records    rows = []    t0 = time.time()    for i, rec in enumerate(items):        pred = generate_answer(rec["input"])        task = rec.get("task") or infer_task(rec["input"])        passage = extract_passage(rec["input"])        row = {"task": task, "subtype": rec.get("subtype"),               "country": rec.get("country"), "pred": pred, "ref": rec["output"]}        if task == "classify":            got, want = parse_labels(pred), parse_labels(rec["output"])            row["json_ok"] = got is not None            if got and want:                for k in ("country", "theme", "terrain", "traveler_profile"):                    row[f"hit_{k}"] = (str(got.get(k)).strip().lower()                                       == str(want.get(k)).strip().lower())                row["pred_theme"] = str(got.get("theme"))                row["true_theme"] = str(want.get("theme"))        elif task in GROUNDED_TASKS:            row["grounded"] = is_grounded(pred, passage)            row["rouge_l"] = rouge_l(pred, rec["output"])            row["refused"] = bool(RE_REFUSAL.search(pred))            # The gold answer being a refusal means this item is really a            # refusal item, whatever the tag says.            row["gold_refusal"] = bool(RE_REFUSAL.search(str(rec["output"])))        elif task == "refuse":            row["refused"] = bool(RE_REFUSAL.search(pred))            row["gold_refusal"] = True        rows.append(row)        if i < verbose_examples:            print(f"\n[{label}] {task}")            print("  prompt  :", rec["input"][:150].replace("\n", " "), "...")            print("  expected:", str(rec["output"])[:140].replace("\n", " "))            print("  got     :", pred[:140].replace("\n", " "))        if (i + 1) % 10 == 0:            print(f"  {i + 1}/{len(items)}  ({time.time() - t0:.0f}s)")    df = pd.DataFrame(rows)    s = {"model": label, "n": len(df)}    cl = df[df.task == "classify"] if "task" in df else df.iloc[0:0]    if len(cl):        s["classify_json_ok"] = round(cl.json_ok.mean(), 3)        for k in ("country", "theme", "terrain", "traveler_profile"):            col = f"hit_{k}"            if col in cl:                s[f"classify_{k}"] = round(cl[col].fillna(False).mean(), 3)        ok = cl.dropna(subset=["pred_theme", "true_theme"]) \            if "pred_theme" in cl else cl.iloc[0:0]        if len(ok):            try:                from sklearn.metrics import f1_score                s["classify_theme_macroF1"] = round(                    f1_score(ok.true_theme, ok.pred_theme,                             average="macro", zero_division=0), 3)            except Exception:                pass    gen = df[df.task.isin(GROUNDED_TASKS)] if "task" in df else df.iloc[0:0]    if len(gen):        # .astype(bool) matters: the column holds Python bools and Nones, so        # pandas infers object dtype and `~` on it raises.        answerable = gen[~gen["gold_refusal"].fillna(False).astype(bool)] \            if "gold_refusal" in gen else gen        g = gen.grounded.dropna()        s["grounded"] = round(g.mean(), 3) if len(g) else None        s["rouge_l"] = round(gen.rouge_l.mean(), 3)        if len(answerable):            s["false_refusal"] = round(answerable.refused.mean(), 3)    sm = df[df.task == "summarize"] if "task" in df else df.iloc[0:0]    if len(sm):        gs = sm.grounded.dropna()        s["summarize_grounded"] = round(gs.mean(), 3) if len(gs) else None        s["summarize_rouge_l"] = round(sm.rouge_l.mean(), 3)    # Anything whose gold output is a refusal, however it was tagged.    if "gold_refusal" in df:        rf = df[df.gold_refusal.fillna(False).astype(bool)]        if len(rf):            s["refusal_recall"] = round(rf.refused.mean(), 3)    print(f"\n=== {label} ===")    for k, v in s.items():        print(f"  {k:26s} {v}")    return s, df

## 8. Baseline — score the model before touching itThis is the cell capstones skip. Do not skip it.Generation is the slow part: roughly 10–20 seconds per item on a T4, so 20items is five to seven minutes. Raise `EVAL_N` for the final write-up.

In [ ]:
eval_records = test_full.to_dict("records")random.Random(SEED).shuffle(eval_records)baseline_summary, baseline_rows = evaluate(eval_records, "base (no LoRA)", n=EVAL_N)baseline_rows.to_csv(f"{DRIVE}/eval_baseline_{RUN_NAME}.csv", index=False)

## 9. LoRA fine-tuning`enable_lora` freezes the backbone and adds the low-rank adapters. Keep theepoch count low: 1B models over a narrow dataset lose instruction-followingand refusal behaviour before they gain much else.

In [ ]:
model.backbone.enable_lora(rank=LORA_RANK)model.preprocessor.sequence_length = SEQUENCE_LENGTHmodel.optimizer.learning_rate = LEARNING_RATEtrainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)total = sum(int(np.prod(w.shape)) for w in model.weights)print(f"LoRA rank {LORA_RANK}")print(f"trainable {trainable:,} of {total:,}  ({100 * trainable / total:.2f}%)")print("lr", float(model.optimizer.learning_rate))

In [ ]:
PROBE = (    "[ANSWER] Answer the question using only the passage below. "    "If the passage does not contain the answer, say so plainly.\n\n"    "QUESTION: How much does it cost to visit?\n\nPASSAGE:\n"    "Kakum National Park, in the Central Region of Ghana, covers 375 square "    "kilometres of tropical forest. It is one of only three places in Africa "    "with a canopy walkway, 350 metres long and linking seven tree tops.")history = []for epoch in range(NUM_EPOCHS):    print(f"\n===== epoch {epoch + 1}/{NUM_EPOCHS} =====")    h = model.fit(data, epochs=1, batch_size=BATCH_SIZE, verbose=1)    history.append({k: float(v[-1]) for k, v in h.history.items()})    # The probe asks for a fee the passage does not give. The right answer is    # to say so. If it starts inventing a price, stop and reconsider.    print("\nprobe (correct answer = 'the passage does not say'):")    print(" ", generate_answer(PROBE)[:300])print("\n", pd.DataFrame(history))

## 10. Score the fine-tuned model

In [ ]:
tuned_summary, tuned_rows = evaluate(eval_records, f"LoRA {RUN_NAME}", n=EVAL_N)tuned_rows.to_csv(f"{DRIVE}/eval_tuned_{RUN_NAME}.csv", index=False)

In [ ]:
comparison = pd.DataFrame([baseline_summary, tuned_summary]).set_index("model")display(comparison.T)metrics = [m for m in ("classify_json_ok", "classify_theme", "classify_country",                       "grounded", "rouge_l", "summarize_grounded",                       "refusal_recall")           if m in comparison.columns and comparison[m].notna().all()]if metrics:    base_v = [comparison.iloc[0][m] for m in metrics]    tuned_v = [comparison.iloc[1][m] for m in metrics]    x = np.arange(len(metrics))    plt.figure(figsize=(9, 3.6))    plt.bar(x - 0.2, base_v, 0.4, label="before", color="#9ca3af")    plt.bar(x + 0.2, tuned_v, 0.4, label="after LoRA", color="#2563eb")    plt.xticks(x, metrics, rotation=20, ha="right")    plt.ylim(0, 1)    plt.ylabel("score")    plt.title(f"Before vs after - {RUN_NAME}  (n={EVAL_N})")    plt.legend()    plt.tight_layout()    plt.show()else:    print("not enough shared metrics to plot - raise EVAL_N")

**Read `false_refusal` alongside `refusal_recall`.** A model that refuseseverything scores a perfect `refusal_recall` and is useless. If`false_refusal` climbs after training, you have over-weighted the refusalslice — cut it in the next data build rather than reaching for the learningrate.

## 11. Invention ratePrompts about places not in the corpus: invented names, real places faroutside West Africa, and real West African sites with no passage attached.The correct behaviour in every case is to decline, because there is no sourceto ground an answer in.

In [ ]:
INVENTED = ["Olumbe Rock Shelters", "Kanjiro Salt Flats", "Tembika Royal Baths",            "Lake Wusuma", "Nzeribe Terraces", "Fort Adanwomase",            "The Gidanwa Caves", "Mbakwe Sacred Pools", "Zarenga Cliff Tombs",            "Opokun Heritage Village"]ELSEWHERE = ["the Eiffel Tower", "Angkor Wat", "Machu Picchu", "Table Mountain",             "the Pyramids of Giza", "Petra", "the Taj Mahal", "Uluru"]REAL_NO_PASSAGE = ["Osun-Osogbo Sacred Grove", "Elmina Castle", "Sukur",                   "the Bandiagara Escarpment", "Kakum National Park"]prompts_adv = []for n in INVENTED:    prompts_adv.append((f"What is {n}?", "invented"))    prompts_adv.append((f"Tell me what there is to see at {n}.", "invented"))for n in ELSEWHERE:    prompts_adv.append((f"What is {n}?", "outside_region"))for n in REAL_NO_PASSAGE:    prompts_adv.append((f"What is {n}?", "real_no_passage"))random.Random(SEED).shuffle(prompts_adv)prompts_adv = prompts_adv[:ADVERSARIAL_N]TEMPLATE = ("[ANSWER] Answer the question using only the passage below. "            "If the passage does not contain the answer, say so plainly.\n\n"            "QUESTION: {q}\n\nPASSAGE:\n(none available)")adv_rows = []for i, (q, kind) in enumerate(prompts_adv):    out = generate_answer(TEMPLATE.format(q=q), max_length=320)    adv_rows.append({"kind": kind, "question": q, "response": out,                     "declined": bool(RE_REFUSAL.search(out))})    if (i + 1) % 10 == 0:        print(f"  {i + 1}/{len(prompts_adv)}")adv = pd.DataFrame(adv_rows)adv["invented"] = ~adv["declined"]print("\ninvention rate by kind (lower is better):")print(adv.groupby("kind")["invented"].mean().round(3).to_string())print(f"\noverall: {adv['invented'].mean():.1%}")adv.to_csv(f"{DRIVE}/eval_adversarial_{RUN_NAME}.csv", index=False)print("\n--- sample responses to invented places ---")for _, r in adv[adv.kind == "invented"].head(4).iterrows():    print(f"\n[{'INVENTED' if r.invented else 'declined'}] {r.question}\n  {r.response[:220]}")

## 12. Save the adapter and log the run

In [ ]:
weights_path = f"{DRIVE}/ijeoma_lora_{RUN_NAME}.weights.h5"model.save_weights(weights_path)print("saved", weights_path)

In [ ]:
log_path = f"{DRIVE}/ijeoma_experiment_log.csv"entry = {    "timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),    "run_name": RUN_NAME,    "preset": PRESET,    "dataset": DATASET,    "smoke_test": SMOKE_TEST,    "n_train": len(prompts),    "lora_rank": LORA_RANK,    "learning_rate": LEARNING_RATE,    "epochs": NUM_EPOCHS,    "batch_size": BATCH_SIZE,    "sequence_length": SEQUENCE_LENGTH,    "mixed_precision": MIXED_PRECISION,    "final_loss": history[-1].get("loss") if history else None,    "eval_n": EVAL_N,    "invention_rate": round(float(adv["invented"].mean()), 3),}for k, v in tuned_summary.items():    if k != "model":        entry[f"tuned_{k}"] = vfor k, v in baseline_summary.items():    if k != "model":        entry[f"base_{k}"] = vrow = pd.DataFrame([entry])if os.path.exists(log_path):    row = pd.concat([pd.read_csv(log_path), row], ignore_index=True)row.to_csv(log_path, index=False)print("logged to", log_path)display(row.tail(3))

## What to do with the numbers**If the smoke test ran clean**, set `SMOKE_TEST = False`, raise `EVAL_N` to100 or more, and do the three runs: rank 8 at 2e-4 for 2 epochs, rank 16 at1e-4 for 3 epochs, then the better of the two on the full merged dataset.**If `grounded` did not improve over baseline**, the problem is the data, notthe hyperparameters. Look at which task is dragging — that is why scoring issplit by task.**If `false_refusal` rose sharply**, there is too much refusal in the mix.Rebuild with `--n-abstain` and `--n-scope` lowered.**If the invention rate is high on `real_no_passage`**, the model is recitingfrom pretraining rather than reading the prompt. That is the failure thegrounded design exists to prevent, and it is worth a paragraph in yourwrite-up whichever way it goes.Every run appends to `ijeoma_experiment_log.csv`. Change one thing at a timeand the log becomes your results section.